In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

# Set up visual style for professional charts
sns.set_theme(style="whitegrid")

print("Loading data...")
# 1. Load the dataset
df = pd.read_csv('../data/raw/raw_analyst_ratings.csv')

# Clean up any 'Unnamed: 0' index columns that might exist
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# ---------------------------------------------------------
# 1. DESCRIPTIVE STATISTICS (Headline Lengths)
# ---------------------------------------------------------
print("\n--- 1. Headline Length Statistics ---")
df['headline_length'] = df['headline'].astype(str).apply(len)
print(df['headline_length'].describe())

plt.figure(figsize=(10, 5))
sns.histplot(df['headline_length'], bins=50, kde=True, color='indigo')
plt.title('Distribution of Headline Character Counts')
plt.xlabel('Number of Characters')
plt.ylabel('Frequency')
plt.show()

# ---------------------------------------------------------
# 2. PUBLISHER ANALYSIS
# ---------------------------------------------------------
print("\n--- 2. Top 10 Publishers ---")
top_publishers = df['publisher'].value_counts().head(10)
print(top_publishers)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_publishers.index, y=top_publishers.values, palette='viridis')
plt.title('Top 10 Most Active Publishers')
plt.xticks(rotation=45)
plt.ylabel('Number of Articles')
plt.show()

# ---------------------------------------------------------
# 3. TIME SERIES ANALYSIS (Trends & Spikes)
# ---------------------------------------------------------
print("\n--- 3. Time Series Analysis ---")
print("Parsing dates (this might take a minute)...")
# Convert date column safely, forcing errors to NaT (Not a Time)
df['date'] = pd.to_datetime(df['date'], errors='coerce', utc=True)
df_time = df.dropna(subset=['date']).copy()

# Visual 3A: Daily News Volume
daily_volume = df_time.groupby(df_time['date'].dt.date).size()

plt.figure(figsize=(14, 6))
daily_volume.plot(color='crimson', linewidth=1.5)
plt.title('Daily News Volume Over Time')
plt.xlabel('Date')
plt.ylabel('Article Count')
plt.grid(True, alpha=0.5)
plt.show()

# Visual 3B: Publishing Times (Hour of the Day)
df_time['hour'] = df_time['date'].dt.hour
hourly_volume = df_time['hour'].value_counts().sort_index()

plt.figure(figsize=(10, 5))
sns.barplot(x=hourly_volume.index, y=hourly_volume.values, color='steelblue')
plt.title('News Volume by Hour of Day (UTC)')
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Number of Articles')
plt.show()

# ---------------------------------------------------------
# 4. TEXT ANALYSIS (Topic Modeling / Keywords)
# ---------------------------------------------------------
print("\n--- 4. Text Analysis (Keywords & Phrases) ---")
print("Analyzing headline text...")

# Look at single words and two-word phrases, excluding common English stop words
vectorizer = CountVectorizer(stop_words='english', ngram_range=(1, 2), max_features=20)

# Fit the vectorizer on the headlines
X = vectorizer.fit_transform(df['headline'].dropna().astype(str))

# Create a DataFrame of the top terms
words = vectorizer.get_feature_names_out()
counts = X.sum(axis=0).A1
word_freq = pd.DataFrame({'Keyword/Phrase': words, 'Frequency': counts})
word_freq = word_freq.sort_values(by='Frequency', ascending=False)

# Visual 4: Top Keywords
plt.figure(figsize=(12, 6))
sns.barplot(data=word_freq, x='Frequency', y='Keyword/Phrase', palette='magma')
plt.title('Top 20 Keywords and Phrases in Financial Headlines')
plt.show()

print("\nEDA COMPLETE! ✅")